In [ ]:
train='/content/train.csv'
test='/content/test.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv(train)
df.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 25 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   age                                 700000 non-null  int64  
 1   alcohol_consumption_per_week        700000 non-null  int64  
 2   physical_activity_minutes_per_week  700000 non-null  int64  
 3   diet_score                          700000 non-null  float64
 4   sleep_hours_per_day                 700000 non-null  float64
 5   screen_time_hours_per_day           700000 non-null  float64
 6   bmi                                 700000 non-null  float64
 7   waist_to_hip_ratio                  700000 non-null  float64
 8   systolic_bp                         700000 non-null  int64  
 9   diastolic_bp                        700000 non-null  int64  
 10  heart_rate                          700000 non-null  int64  
 11  cholesterol_total         

### Exploratory Data Analysis

### Model Training

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_predict

from lightgbm import LGBMClassifier

In [ ]:
X = df.drop("diagnosed_diabetes", axis=1)
y = df["diagnosed_diabetes"].astype(int)

In [ ]:
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

In [ ]:
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features)
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

In [ ]:
for df_ in [X_train, X_test]:
    df_["bp_ratio"] = df_["systolic_bp"] / df_["diastolic_bp"]
    df_["activity_bmi"] = df_["physical_activity_minutes_per_week"] / df_["bmi"]

In [ ]:
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,  #0.07
    max_depth=5,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=0.2,
    class_weight="balanced",
    random_state=42,
    device_type="gpu"
)

In [ ]:
pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['id', 'age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi',
       'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
       'cholesterol_total', 'hd...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['gender', 'ethnicity', 'education_level', 'income_level',
       'smoking_status', 'employment_status'],
      dtype='object'))])),
                ('model',
                 LGBMClassifier(class_weight='balanced', colsample_bytree=0.85,
                                device_type='gpu', learning_rate=0.05,
                                max_depth=5, n_estimators=500, random_state=42,
                                reg_alpha=0.2, reg_lambda=0.2,
                                subsample=0.85))])

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:,1]

print("ROC AUC:", roc_auc_score(y_test, y_proba))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC AUC: 0.7236522976376933


In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=900,
    depth=7,
    learning_rate=0.04,
    l2_leaf_reg=6,
    loss_function='Logloss',
    eval_metric='AUC',
    task_type='GPU',
    devices='0',
    verbose=False
)

cat_model.fit(X_train, y_train, cat_features=list(cat_features))

Default metric period is 5 because AUC is/are not implemented for GPU


In [ ]:
y_proba=cat_model.predict_proba(X_test)[:,1]
print("ROC AUC:", roc_auc_score(y_test, y_proba))

ROC AUC: 0.7224196025805785


In [ ]:
lgb_proba = pipeline.predict_proba(X_test)[:,1]
cat_proba = cat_model.predict_proba(X_test)[:,1]

final_proba = 0.55 * lgb_proba + 0.45 * cat_proba

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Testing

In [ ]:
df_test=pd.read_csv(test)

In [ ]:
id=df_test['id']

In [ ]:
df_test["bp_ratio"] = df_test["systolic_bp"] / df_test["diastolic_bp"]
df_test["activity_bmi"] = df_test["physical_activity_minutes_per_week"] / df_test["bmi"]

In [ ]:
# y_proba = pipeline.predict_proba(df_test)[:,1]
lgb_proba = pipeline.predict_proba(df_test)[:,1]
cat_proba = cat_model.predict_proba(df_test)[:,1]

y_proba = 0.55 * lgb_proba + 0.45 * cat_proba

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
submission=pd.DataFrame({'id':id,'diagnosed_diabetes':y_proba})
submission.to_csv('submission.csv',index=False)